# Building micrograd

Environment check — run this first to confirm the kernel and libraries are wired up correctly.

In [6]:
import sys
import numpy as np
import matplotlib
import torch
import graphviz
import math 

# print(f"python    : {sys.version.split()[0]}")
# print(f"venv      : {sys.prefix}")
# print(f"numpy     : {np.__version__}")
# print(f"matplotlib: {matplotlib.__version__}")
# print(f"torch     : {torch.__version__}")
# print(f"graphviz  : {graphviz.__version__}")

In [8]:
class Value(): 

    def __init__(self, data, _children = (), _op = '', label = ''): 
        self.data = data 
        self.grad = 0.0 
        self._backward = lambda:None 
        self._prev = set(_children) 
        self._op = _op 
        self.label = label 
    
    def __repr__(self): 
        out = f"Value(data = {self.data})" 
        return out 
    
    def __add__(self, other): 
        other = other if isinstance(other, Value) else Value(other) 
        out = Value(self.data + other.data, (self, other), label = '+') 

        def _backward(): 
            self.grad += 1.0 * out.grad 
            other.grad += 1.0 * out.grad 
        out._backward = _backward 

        return out 
    
    def __radd__(self, other): 
        return self + other 

    def __mul__(self, other): 
        other = other if isinstance(other, Value) else Value(other) 
        out = Value(self.data * other.data, (self, other), label = '*') 

        def _backward(): 
            self.grad += other.data * out.grad 
            other.grad += self.data * out.grad 
        out._backward = _backward 

        return out 
    
    def __rmul__(self, other):
        return self * other 

    def __sub__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = self + (-1.0 * other)

        return out 

    def __rsub__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self - other 
    
    def __pow__(self, other):
        assert(isinstance(other, (int, float)))
        out = Value(self.data ** other, (self, ), f'**{other}')

        def _backward():
            self.grad += out.grad * (other * (self.data ** (other - 1.0)))
        out._backward = _backward

        return out 
    
    def __rpow__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return pow(other, self)
    
    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * (other.data ** (-1.0)), (self, other), label = '/')
        
        def _backward():
            self.grad += (other.data ** (-1.0)) * out.grad 
            other.grad += (self.data * math.log(other.data)) * out.grad
        out._backward = _backward 

        return out 
    
    def __rtruediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return other / self

    def exp(self):
        out = Value(math.exp(self.data), (self, ), label = 'e')

        def _backward():
            self.grad += math.exp(self.data) * out.grad 
        out._backward = _backward

        return out 
    
    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0

        for node in reversed(topo):
            node._backward()
    